# Análise Exploratória de Dados — Carga de Energia do ONS

Este projeto realiza uma análise exploratória dos dados de carga diária de energia elétrica disponibilizados pelo Operador Nacional do Sistema Elétrico (ONS), abrangendo o período entre 2015 e 2025.

O objetivo da análise é compreender o comportamento temporal da carga de energia nos subsistemas brasileiros, identificar padrões, tendências e possíveis anomalias nos dados.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

## Carregamento dos dados

In [2]:
arquivos = [
    "../CsvDados/CARGA_ENERGIA_2015.csv",
    "../CsvDados/CARGA_ENERGIA_2016.csv",
    "../CsvDados/CARGA_ENERGIA_2017.csv",
    "../CsvDados/CARGA_ENERGIA_2018.csv",
    "../CsvDados/CARGA_ENERGIA_2019.csv",
    "../CsvDados/CARGA_ENERGIA_2020.csv",
    "../CsvDados/CARGA_ENERGIA_2021.csv",
    "../CsvDados/CARGA_ENERGIA_2022.csv",
    "../CsvDados/CARGA_ENERGIA_2023.csv",
    "../CsvDados/CARGA_ENERGIA_2024.csv",
    "../CsvDados/CARGA_ENERGIA_2025.csv",
]

df = pd.concat(
    [pd.read_csv(arq, sep=';', encoding='utf-8') for arq in arquivos],
    ignore_index=True
)

print(f'✅ {df.shape[0]:,} linhas carregadas')

✅ 16,072 linhas carregadas


## Análise inicial da estrutura dos dados

In [3]:
df.head() 

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiamwmed
0,N,Norte,2015-01-01,4541.005167
1,NE,Nordeste,2015-01-01,8308.521949
2,S,Sul,2015-01-01,7717.620040
3,SE,Sudeste/Centro-Oeste,2015-01-01,30874.778708
4,N,Norte,2015-01-02,4886.834250


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16072 entries, 0 to 16071
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_subsistema          16072 non-null  str    
 1   nom_subsistema         16072 non-null  str    
 2   din_instante           16072 non-null  str    
 3   val_cargaenergiamwmed  16068 non-null  float64
dtypes: float64(1), str(3)
memory usage: 502.4 KB


In [5]:
df.describe()

,val_cargaenergiamwmed
count,16068.000000
mean,16964.393600
std,13103.657723
min,3969.773875
25%,8408.050559
50%,11260.678479
75%,21024.181104
max,55584.896708


Desvio padrão alto, uma causa é a disparidade da demanda energética entre as regiões

In [6]:
df.isnull().sum()

id_subsistema            0
nom_subsistema           0
din_instante             0
val_cargaenergiamwmed    4
dtype: int64

In [7]:
df[df['val_cargaenergiamwmed'].isnull()]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiamwmed
392,N,Norte,2015-04-09,NaN
393,NE,Nordeste,2015-04-09,NaN
394,S,Sul,2015-04-09,NaN
395,SE,Sudeste/Centro-Oeste,2015-04-09,NaN


Foram identificados valores ausentes no dia 09/04/2015 para todos os subsistemas. Como os dados anteriores e posteriores estavam disponíveis e consistentes, optou-se por utilizar interpolação linear para estimar os valores faltantes.

## Tratamento de valores ausentes

In [8]:
# Converter data
df['din_instante'] = pd.to_datetime(df['din_instante'])

# Ordenar por subsistema e data — essencial para a interpolação funcionar corretamente
df = df.sort_values(['id_subsistema', 'din_instante']).reset_index(drop=True)

# Interpolar dentro de cada subsistema separadamente
df['val_cargaenergiamwmed'] = (
    df.groupby('id_subsistema')['val_cargaenergiamwmed']
    .transform(lambda x: x.interpolate(method='linear'))
)

# Confirmar que os nulos foram preenchidos
nulos_restantes = df['val_cargaenergiamwmed'].isnull().sum()
print(f'✅ Nulos restantes: {nulos_restantes}')

# Verificar o dia que tinha nulos
print('\n📅 Dia 09/04/2015 após interpolação:')
print(df[df['din_instante'] == '2015-04-09'][['id_subsistema', 'din_instante', 'val_cargaenergiamwmed']])

✅ Nulos restantes: 0

📅 Dia 09/04/2015 após interpolação:
      id_subsistema din_instante  val_cargaenergiamwmed
98                N   2015-04-09            5388.354417
4116             NE   2015-04-09           10523.588220
8134              S   2015-04-09           11472.557228
12152            SE   2015-04-09           37295.256378


Os valores ausentes foram tratados utilizando interpolação linear dentro de cada subsistema separadamente. Essa abordagem preserva o comportamento temporal específico de cada região do sistema elétrico brasileiro, evitando distorções causadas por médias globais.

In [9]:
df.dtypes

id_subsistema                       str
nom_subsistema                      str
din_instante             datetime64[us]
val_cargaenergiamwmed           float64
dtype: object

In [10]:
print(df.isnull().sum())

id_subsistema            0
nom_subsistema           0
din_instante             0
val_cargaenergiamwmed    0
dtype: int64


In [11]:
print(f'Duplicados encontrados: {df.duplicated().sum()}')

Duplicados encontrados: 0


In [12]:
df['id_subsistema'].unique()

<StringArray>
['N', 'NE', 'S', 'SE']
Length: 4, dtype: str

In [13]:
df.describe()

,din_instante,val_cargaenergiamwmed
count,16072,16072.000000
mean,2020-07-01 12:00:00,16964.195876
min,2015-01-01 00:00:00,3969.773875
25%,2017-10-01 00:00:00,8408.050559
50%,2020-07-01 12:00:00,11260.678479
75%,2023-04-02 00:00:00,21024.181104
max,2025-12-31 00:00:00,55584.896708
std,NaN,13103.496671


## Analise dos Dados

In [14]:
# Linha com menor valor
minimo = df.loc[df['val_cargaenergiamwmed'].idxmin()]

print('🔽 Menor valor encontrado:')
print(minimo)

# Linha com maior valor
maximo = df.loc[df['val_cargaenergiamwmed'].idxmax()]

print('\n🔼 Maior valor encontrado:')
print(maximo)

🔽 Menor valor encontrado:
id_subsistema                             NE
nom_subsistema                      Nordeste
din_instante             2018-08-25 00:00:00
val_cargaenergiamwmed            3969.773875
Name: 5350, dtype: object

🔼 Maior valor encontrado:
id_subsistema                              SE
nom_subsistema           Sudeste/Centro-Oeste
din_instante              2025-02-18 00:00:00
val_cargaenergiamwmed            55584.896708
Name: 15755, dtype: object


In [15]:
df['ano'] = df['din_instante'].dt.year
df['mes'] = df['din_instante'].dt.month
df['dia_semana'] = df['din_instante'].dt.dayofweek # 0=Segunda, 6=Domingo
df['is_weekend'] = df['dia_semana'].isin([5, 6])

Foram criadas variáveis temporais derivadas da coluna de data, permitindo análises sazonais e comportamentais da carga energética ao longo dos anos, meses e dias da semana. Essas variáveis auxiliam na identificação de padrões temporais e possíveis anomalias no consumo de energia.

In [16]:
df[
    (df['nom_subsistema'] == 'Nordeste') &
    (df['din_instante'].between('2018-08-20', '2018-08-30'))
][['din_instante', 'val_cargaenergiamwmed']]

,din_instante,val_cargaenergiamwmed
5345,2018-08-20,10141.094292
5346,2018-08-21,10281.005000
5347,2018-08-22,10275.046333
5348,2018-08-23,10348.784458
5349,2018-08-24,10451.198208
5350,2018-08-25,3969.773875
5351,2018-08-26,9196.706042
5352,2018-08-27,10129.237208
5353,2018-08-28,10338.145917
5354,2018-08-29,10199.799542


A análise temporal do menor valor registrado revelou comportamento inconsistente em relação aos dias adjacentes. Enquanto os valores do subsistema Nordeste permaneceram próximos de 10 mil MWmed antes e após 25/08/2018, foi identificado um valor abruptamente inferior de aproximadamente 4 mil MWmed nessa data.

Esse comportamento sugere a presença de um possível outlier temporal, potencialmente associado a falha de medição, inconsistência operacional ou evento excepcional no sistema elétrico.

In [17]:
df[
    (df['nom_subsistema'] == 'Sudeste/Centro-Oeste') &
    (df['din_instante'].between('2025-02-13', '2025-02-23'))
][['din_instante', 'val_cargaenergiamwmed']]

,din_instante,val_cargaenergiamwmed
15750,2025-02-13,53428.736708
15751,2025-02-14,52974.155083
15752,2025-02-15,49033.897375
15753,2025-02-16,46368.360792
15754,2025-02-17,54572.624875
15755,2025-02-18,55584.896708
15756,2025-02-19,54696.465042
15757,2025-02-20,55196.378625
15758,2025-02-21,54703.323833
15759,2025-02-22,49920.815333


A análise do maior valor registrado no dataset indica comportamento consistente com os dias adjacentes. Entre 13/02/2025 e 23/02/2025, o subsistema Sudeste/Centro-Oeste apresentou valores elevados e relativamente estáveis, variando entre aproximadamente 46 mil e 55 mil MWmed.

O pico observado em 18/02/2025 (55.584 MWmed) não representa uma ruptura abrupta na série temporal, mas sim um aumento gradual dentro de um período de alta demanda energética na região.

Esse comportamento sugere que o valor máximo identificado possui maior probabilidade de representar um evento real do sistema elétrico, e não uma inconsistência ou erro de medição.

In [ ]:
# Média anual por subsistema
carga_anual = (
    df.groupby(['ano', 'nom_subsistema'])['val_cargaenergiamwmed']
    .mean()
    .reset_index()
)